In [1]:
import os
import pandas as pd
import re, string, nltk
from datasets import load_dataset
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk import word_tokenize, pos_tag
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
import time

c:\Users\ysnxlmted\Projects\documents_classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Создание папки для nltk данных, если её нет
nltk_data_dir = os.path.expanduser('../nltk_data')
if not os.path.exists(nltk_data_dir):
    os.makedirs(nltk_data_dir)

# Добавление пути в nltk
nltk.data.path.append(nltk_data_dir)

# Загрузка всех необходимых ресурсов
resources = ['punkt', 'wordnet', 'omw-1.4', 'punkt_tab', 
             'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'stopwords']

for resource in resources:
    try:
        nltk.download(resource, download_dir=nltk_data_dir, quiet=False)
    except:
        print(f"Ресурс {resource} уже загружен или произошла ошибка")

[nltk_data] Downloading package punkt to ../nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to ../nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to ../nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to ../nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     ../nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     ../nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to ../nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[{}[\]()<>]', '', text)
    text = re.sub(r'\d+\.\d+\.\d+\.\d+', '', text)
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    text = re.sub(email_pattern, '', text)
    text = re.sub(r'\S+/\S+/\S+', '', text)
    text = re.sub(r'\S+\.\S+/\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[`"\']{2,}', ' ', text)
    text = re.sub(r'[`"\']', ' ', text)
    text = re.sub(r'[\[\]{}()<>]', ' ', text)
    text = re.sub(r'(\w+)-(\d+)', r'\1\2', text)
    text = re.sub(r'(\d+)-(\w+)', r'\1\2', text)
    smile_pattern = r'[:;=][\-^]?[)D\(\[\]pP]+'
    text = re.sub(smile_pattern, '', text)
    text = re.sub(r'([!?.,])\1+', r'\1', text)
    text = re.sub(r'-{2,}', ' ', text)
    text = re.sub(r'\+?\d[\d\s\-\(\)]{7,}\d', '', text)
    text = re.sub(r'[^\w\s]', ' ', text) 
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

stop_words = set(stopwords.words("english"))
additional_stops = {
    'would', 'could', 'should', 'might', 'may', 'get', 'go', 'see', 
    'know', 'like', 'want', 'need', 'say', 'think', 'come', 'take',
    'use', 'make', 'well', 'also', 'even', 'many', 'much', 'still',
    'however', 'though', 'although', 'since', 'yet', 'already'
}
stop_words.update(additional_stops)

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

lemmatizer = nltk.WordNetLemmatizer()

def preprocess_with_lemmatization(text, pos_filter="all"):
    tokens = word_tokenize(clean_text(text))
    tagged = pos_tag(tokens)
    lemmatized_tokens = []
    for token, tag in tagged:
        if token in stop_words or token in string.punctuation:
            continue
        is_noun_adj = tag.startswith('N') or tag.startswith('J')
        if pos_filter == "nouns_adj" and not is_noun_adj:
            continue
        lemmatized_tokens.append(lemmatizer.lemmatize(token, get_wordnet_pos(tag)))
    return " ".join(lemmatized_tokens)

stemmer = PorterStemmer()
def preprocess_with_stemming(text):
    tokens = word_tokenize(clean_text(text))
    res = []
    for token in tokens:
        if token in stop_words or token in string.punctuation: continue
        res.append(stemmer.stem(token))
    return " ".join(res)

In [4]:
from datasets import load_dataset
import pandas as pd

# 1. Загрузка
dataset = load_dataset("SetFit/20_newsgroups")
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

# 2. Полный список категорий (порядок фиксирован для 20 newsgroups)
all_categories = [
    'alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 
    'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 
    'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 
    'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 
    'sci.space', 'soc.religion.christian', 'talk.politics.guns', 
    'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc'
]

# 3. Выбираем нужные 4 категории
selected_categories = [    
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware", 
    "comp.graphics",
    "comp.windows.x"]

# 4. Находим их ID (индексы в общем списке)
selected_ids = [all_categories.index(cat) for cat in selected_categories]

# 5. Фильтрация по ID
train_df = train_df[train_df['label'].isin(selected_ids)].reset_index(drop=True)
test_df = test_df[test_df['label'].isin(selected_ids)].reset_index(drop=True)

# 6. Перемаппинг лейблов в 0, 1, 2, 3
label_map = {old_id: new_id for new_id, old_id in enumerate(selected_ids)}
train_df['label'] = train_df['label'].map(label_map)
test_df['label'] = test_df['label'].map(label_map)

print(f"✅ Классы: {selected_categories}")
print(f"📊 Train: {len(train_df)}, Test: {len(test_df)}")

Repo card metadata block was not found. Setting CardData to empty.


✅ Классы: ['comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.graphics', 'comp.windows.x']
📊 Train: 2345, Test: 1561


## 1 часть

In [5]:
import numpy as np

X_train_raw = train_df['text']
y_train = train_df['label'].to_numpy()

X_test_raw = test_df['text']
y_test = test_df['label'].to_numpy()

In [6]:
X_train_tokens = [preprocess_with_lemmatization(text) for text in X_train_raw]
X_test_tokens = [preprocess_with_lemmatization(text) for text in X_test_raw]

In [7]:
X_train_tokens = [preprocess_with_lemmatization(text).split() for text in X_train_raw]
X_test_tokens = [preprocess_with_lemmatization(text).split() for text in X_test_raw]
X_train_text = [" ".join(tokens) for tokens in X_train_tokens]
X_test_text = [" ".join(tokens) for tokens in X_test_tokens]

tfidf = TfidfVectorizer()
tfidf.fit(X_train_text)

w2v = Word2Vec(X_train_tokens, vector_size=100, window=5, min_count=2, workers=1, epochs=10)

vocab = tfidf.vocabulary_

def doc_vec(tokens, tfidf_row):
    vecs = []
    weights = []
    for token in tokens:
        if token in w2v.wv and token in vocab:
            w = tfidf_row[0, vocab[token]]
            if w > 0:
                vecs.append(w2v.wv[token] * w)
                weights.append(w)
    return np.sum(vecs, axis=0) / sum(weights) if weights else np.zeros(w2v.vector_size)

X_train_emb = np.vstack([doc_vec(tokens, tfidf.transform([text])) for tokens, text in zip(X_train_tokens, X_train_text)])
X_test_emb = np.vstack([doc_vec(tokens, tfidf.transform([text])) for tokens, text in zip(X_test_tokens, X_test_text)])


In [8]:
clf = RandomForestClassifier(n_estimators=200, max_depth=30, random_state=42)
clf.fit(X_train_emb, y_train)

y_pred = clf.predict(X_test_emb)
score = f1_score(y_test, y_pred, average='macro')

print(f"F1-score (macro): {score:.4f}")

F1-score (macro): 0.6486


## 2 часть

In [9]:
import torch
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)


def encode(texts, batch_size=32):
    model.eval()
    cls_embeddings = []
    mean_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
        
        with torch.no_grad():
            outputs = model(**inputs)  

        last_hidden_state = outputs.last_hidden_state
        
        # cls embeddings
        cls_emb = last_hidden_state[:, 0, :].numpy()
        cls_embeddings.append(cls_emb)

        # mean
        mask = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * mask, 1)
        sum_mask = torch.clamp(mask.sum(1), min=1e-9)
        mean_emb = (sum_embeddings / sum_mask).numpy()
        mean_embeddings.append(mean_emb)
    
    return np.vstack(cls_embeddings), np.vstack(mean_embeddings)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2651.92it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
print("Извлечение BERT-эмбеддингов для Train...")
train_cls, train_mean = encode(X_train_raw.tolist())
print("Извлечение BERT-эмбеддингов для Test...")
test_cls, test_mean = encode(X_test_raw.tolist())

Извлечение BERT-эмбеддингов для Train...
Извлечение BERT-эмбеддингов для Test...


In [11]:
def evaluate_rf(train_features, test_features, name):
    rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)
    rf.fit(train_features, y_train)
    preds = rf.predict(test_features)
    score = f1_score(y_test, preds, average='macro')
    print(f"RF on {name} F1-score: {score:.4f}")
    return score

# Сравниваем
evaluate_rf(train_mean, test_mean, "BERT Mean")

RF on BERT Mean F1-score: 0.6252


0.6252181519654207